In [11]:
import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

df = pd.read_csv('final_dataset.csv')

print(f"Датасет загружен: {len(df)} строк.")
print(df['label'].value_counts())

Датасет загружен: 66016 строк.
label
0    33008
1    33008
Name: count, dtype: int64


In [12]:
refusal_phrase = "я не могу обсуждать эту тему. давайте поговорим о чём-нибудь ещё"

def remove_refusal(text):
    clean = re.sub(re.escape(refusal_phrase), '', text, flags=re.IGNORECASE)
    return clean.strip()

df.loc[df['label'] == 1, 'text'] = df.loc[df['label'] == 1, 'text'].apply(remove_refusal)

df = df[df['text'].str.split().str.len() > 20]

In [13]:
print(f"Датасет загружен: {len(df)} строк.")
print(df['label'].value_counts())

Датасет загружен: 58983 строк.
label
0    32770
1    26213
Name: count, dtype: int64


In [14]:
def clean_text(text):
    text = re.sub(r'[^\w\s.,!?;:-—]', '', text)
    return text.lower().strip()

df['uppercase_ratio'] = df['text'].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / len(str(x)) if len(str(x))>0 else 0)
df['content_clean'] = df['text'].apply(clean_text)

In [15]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['punctuation_ratio'] = df['text'].apply(lambda x: sum(1 for c in str(x) if c in '.,!?;:-') / len(str(x)) if len(str(x))>0 else 0)
df['stopword_ratio'] = 0.0 # Оставляем 0, так как в handler.py стоп-слова не вычисляются в фичи
df['unique_word_ratio'] = df['content_clean'].apply(lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0)
df['avg_word_length'] = df['char_count'] / df['word_count']


df = df.fillna(0)
print("Признаки успешно посчитаны.")

Признаки успешно посчитаны.


In [16]:
df = df.sample(frac=1, random_state=12345).reset_index(drop=True)
print(df['label'].value_counts())

label
0    32770
1    26213
Name: count, dtype: int64


In [17]:
df.head(10)

,doc_id,chunk_id,text,word_count,label,source_model,uppercase_ratio,content_clean,char_count,punctuation_ratio,stopword_ratio,unique_word_ratio,avg_word_length
0,sbornik-statisticheskih-svedeniy-o-stavropolsk...,sbornik-statisticheskih-svedeniy-o-stavropolsk...,**Сборник статистических сведений о Ставрополь...,132,1,YandexGPT,0.020282,сборник статистических сведений о ставропольск...,1134,0.018519,0.0,0.765152,8.590909
1,pamyati-l-s-vasilieva,pamyati-l-s-vasilieva_0,Ш МЕМОШЛМ Памяти Л.С. Васильева 6 октября 2016...,284,0,Human,0.043627,ш мемошлм памяти л.с. васильева 6 октября 2016...,2040,0.034804,0.0,0.799296,7.183099
2,matematika-laykov,matematika-laykov_6,Каждый клиент получает в свое распоряжение цен...,279,0,Human,0.018224,каждый клиент получает в свое распоряжение цен...,2140,0.020561,0.0,0.799283,7.670251
3,filosofskiy-fakultet-tomskogo-gosudarstvennogo...,filosofskiy-fakultet-tomskogo-gosudarstvennogo...,Проблемы правового социализма (либерализм и со...,227,0,Human,0.041619,проблемы правового социализма либерализм и соц...,1754,0.038769,0.0,0.811060,7.726872
4,deviantnoe-povedenie-kak-organizovat-profilakt...,deviantnoe-povedenie-kak-organizovat-profilakt...,В рамках профилактики девиантного поведения в ...,268,1,YandexGPT,0.006454,в рамках профилактики девиантного поведения в ...,2324,0.018933,0.0,0.738806,8.671642
5,materialy-po-istorii-rossiyskoy-arheologii-v-s...,materialy-po-istorii-rossiyskoy-arheologii-v-s...,"В работе Серых (2012, с. 124–130) рассматривае...",337,1,YandexGPT,0.047729,"в работе серых 2012, с. 124–130 рассматриваетс...",2598,0.035027,0.0,0.729970,7.709199
6,pifagoreizm-v-sovremennoy-filosofii-matematiki,pifagoreizm-v-sovremennoy-filosofii-matematiki_1,PYTHAGOREANISM IN MODERN PHILOSOPHY OF MATHEMA...,298,0,Human,0.043281,pythagoreanism in modern philosophy of mathema...,1987,0.021641,0.0,0.583893,6.667785
7,nechetkie-modeli-netochnoy-matematiki,nechetkie-modeli-netochnoy-matematiki_1_ai,"В статье Ушакова В. Н., Лахтина А. С. и Лебеде...",91,1,YandexGPT,0.020539,"в статье ушакова в. н., лахтина а. с. и лебеде...",779,0.026958,0.0,0.879121,8.560440
8,smysl-istorii-v-religioznom-i-filosofskom-poni...,smysl-istorii-v-religioznom-i-filosofskom-poni...,"1. Pushkin, A. S. Works in 3 volumes; volume 2...",283,1,YandexGPT,0.079141,"1. pushkin, a. s. works in 3 volumes; volume 2...",1769,0.093838,0.0,0.657244,6.250883
9,sistemnaya-nechetkaya-intervalnaya-matematika-...,sistemnaya-nechetkaya-intervalnaya-matematika-...,11. Модификация метода наименьших квадратов пр...,268,0,Human,0.026525,11. модификация метода наименьших квадратов пр...,1885,0.024403,0.0,0.745174,7.033582


In [18]:
features_to_use = ['char_count', 'word_count', 'punctuation_ratio', 'stopword_ratio', 'unique_word_ratio', 'uppercase_ratio', 'avg_word_length']

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), 'content_clean'),
        ('num', StandardScaler(), features_to_use)
    ]
)

In [19]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', solver='liblinear'))
])

X = df[['content_clean'] + features_to_use]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12345)


model.fit(X_train, y_train)


print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.90      0.90      0.90      6556
           1       0.88      0.87      0.87      5241

    accuracy                           0.89     11797
   macro avg       0.89      0.89      0.89     11797
weighted avg       0.89      0.89      0.89     11797



In [20]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/ai_text_classifier_science.pkl')

print("Модель сохранена в 'models/ai_text_classifier_science.pkl'")

Модель сохранена в 'models/ai_text_classifier_science.pkl'
